In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from sklearn.metrics import (
confusion_matrix,
ConfusionMatrixDisplay,
classification_report,
accuracy_score
)

#Part I

In [ ]:

wineData = load_wine()
X = pd.DataFrame(
wineData.data,
columns=wineData.feature_names
)
y = pd.Series(wineData.target)
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)
display(X.head())
print("this is y counts",y.value_counts().sort_index())

Shape of X: (178, 13)
Shape of y: (178,)


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0


this is y counts 0    59
1    71
2    48
Name: count, dtype: int64


##Questions Part 1

1. How many observations are in the dataset?
178

2. How many input features does each observation have?
13

3. How many classes are there?
3

4. Is this a regression or classification problem?
Classification

#Part II

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.40, random_state=42, stratify=y)
# Second: split the remaining data into validation and test sets
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)
print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

scaler = StandardScaler()
# Fit only on training data
X_train = scaler.fit_transform(X_train)
# Transform validation and test data using the same scaler

X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# Convert the data into PyTorch tensors:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train.values, dtype=torch.long)
y_val = torch.tensor(y_val.values, dtype=torch.long)
y_test = torch.tensor(y_test.values, dtype=torch.long)

Training: (106, 13) (106,)
Validation: (36, 13)
Testing: (36, 13)


##Questions Part II

1. Why do we standardize the features?
To avoid larger quantities in the data from dominating the distribution

2. Why do we fit the scaler using only the training data?
To avoid leakage in our training set. If we scale everything the model would have peeked at the testing data and thus it might prevent the model from generalizing properly, since technically it had seen the testing data
3. What is the purpose of the validation dataset?
It allows us to tune hyper parameters without touching the test data set
4. Why should we keep the test dataset separate?
So the model doesn't learn from it, and we have a more accurate evaluation

#Part III

In [ ]:
class RegressionNN(nn.Module):
    def __init__(self, hidden_sizes):
        super().__init__()
        layers = []
        input_size = 1 # set the input size according to your dataset
        output_size = 1 # set according to the dataset
        for hidden_size in hidden_sizes:
            layers.append(
                nn.Linear(input_size, hidden_size)
                )

            layers.append(nn.Tanh())
            input_size = hidden_size
        # One output neuron because this is regression
        layers.append(nn.Linear(input_size, output_size))
        self.network = nn.Sequential(*layers)
    def forward(self, x):
        return self.network(x)

##Questions Part III

1. Why do we need an activation function?
    The activation function introduces non-linearity to the output. Each layer is multiplied by the weight and we add a bias, since multiplication and addition are linear operations the layers could collapse into a single transformation, which would make the model less accurate. The activation function prevents this from happening.
2. How to calculate the number of input neurons in the first layer?
    The first layer must have the same amount of features